# SAM 2 full runs (Tiny and Large on DAVIS val, SA-V val, SA-V test subset)
Run on a GPU runtime (an A100 or L4 is best; T4 works for Tiny). Order: mount Drive, set the data root, get the repo, install (SAM 2 is built so its CUDA extension matches Colab's own torch), restart, check the environment, download checkpoints and evaluators, build the prompts, run the six model x split runs, check completeness, score, copy the CSVs.

Every run cell is safe to re-run: finished videos are skipped, an unfinished one is wiped and redone. If Colab disconnects, reconnect, re-run the Drive, DATA_ROOT, repo and (if used) local-copy cells, then re-run the run cell that was interrupted.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
DATA_ROOT = ""  # @param {type:"string"}
assert DATA_ROOT, "Fill in DATA_ROOT (the Drive folder that holds the datasets)"
assert os.path.exists(DATA_ROOT), f"DATA_ROOT does not exist: {DATA_ROOT}"
DRIVE_DATA_ROOT = DATA_ROOT
os.environ["SAM2FR_DATA_ROOT"] = DATA_ROOT

In [ ]:
# After this phase is merged, set BRANCH to "main".
BRANCH = "phase-2b-sam2-runs"  # @param {type:"string"}
REPO_URL = "https://github.com/VEDITHREDDY26/sam2-failure-recovery.git"
REPO_DIR = "/content/sam2-failure-recovery"
import os
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch origin
else:
    !git clone {REPO_URL} {REPO_DIR}
!git -C {REPO_DIR} checkout {BRANCH}
!git -C {REPO_DIR} pull origin {BRANCH}
%cd {REPO_DIR}

## Install
Skip these two cells after the restart. The first installs the pinned packages except SAM 2; the second builds SAM 2 against the torch that Colab already has (`--no-build-isolation`, `SAM2_BUILD_ALLOW_ERRORS=0` so a failed CUDA build stops the install instead of being hidden). SAM 2's small-hole filling needs that CUDA extension, and all six runs must use it.

In [ ]:
# Colab ships plain, headless and contrib OpenCV builds that share one cv2 folder;
# remove all three, then install only the pinned contrib build.
!pip uninstall -y -q opencv-python opencv-python-headless opencv-contrib-python
!grep -v '^SAM-2' requirements-colab.txt | grep -v '^#' | grep -v '^$' > /content/requirements-no-sam2.txt
!pip install -q -r /content/requirements-no-sam2.txt
!pip install -q --force-reinstall --no-deps opencv-contrib-python==4.11.0.86
!pip list 2>/dev/null | grep -i opencv

In [ ]:
# SAM 2 at the pinned commit, with its CUDA extension built against Colab's torch.
SAM2_REQ = next(l.strip() for l in open("requirements-colab.txt") if l.startswith("SAM-2"))
print(SAM2_REQ)
!nvcc --version | tail -2
!pip uninstall -y SAM-2 sam2
!pip install -q setuptools wheel ninja
!SAM2_BUILD_ALLOW_ERRORS=0 pip install --no-build-isolation "{SAM2_REQ}"


Restart the session now (Runtime > Restart session), then re-run the Drive, DATA_ROOT and repo cells and continue from the environment check below; skip the install cells. If the check fails, copy its whole output and send it back; do not continue to the runs.

In [ ]:
import json, platform, sys, warnings
import numpy, cv2, torch
print("python", platform.python_version())
print("torch", torch.__version__, "| built for CUDA", torch.version.cuda)
print("cuda available", torch.cuda.is_available())
print("numpy", numpy.__version__, "| cv2", cv2.__version__, "| TrackerCSRT_create", hasattr(cv2, "TrackerCSRT_create"))
if not torch.cuda.is_available():
    raise RuntimeError("No GPU: use Runtime > Change runtime type > GPU, then restart and re-run.")
gpu = torch.cuda.get_device_name(0)
print("gpu", gpu)

# Hole-filling check: the compiled SAM 2 extension must work on a CUDA tensor, with no silent skipping.
try:
    from sam2 import _C  # noqa: F401
    from sam2.utils.misc import fill_holes_in_mask_scores, get_connected_components
    mask = torch.zeros(1, 1, 16, 16, device="cuda")
    mask[..., 2:14, 2:14] = 1.0          # foreground square
    mask[..., 6:8, 6:8] = -1.0           # a 2x2 hole inside it
    labels, areas = get_connected_components(mask <= 0)
    assert int(areas[0, 0, 6, 6]) == 4, f"hole area {int(areas[0, 0, 6, 6])}, expected 4"
    with warnings.catch_warnings():
        warnings.simplefilter("error")   # fill_holes_in_mask_scores only warns on failure; make it raise
        filled = fill_holes_in_mask_scores(mask, 8)
    assert bool((filled[..., 6:8, 6:8] > 0).all()), "the small hole was not filled"
    print("hole filling: OK (connected components on CUDA, 2x2 hole filled with max area 8)")
except Exception as error:
    raise RuntimeError(
        "SAM 2 hole filling does NOT work (CUDA extension missing or broken). Do not run anything else. "
        f"Send the full output of this cell and of the SAM 2 install cell back. Error: {error!r}"
    ) from error

import sam2
from pathlib import Path
env = {"torch": torch.__version__, "cuda": torch.version.cuda, "gpu": gpu, "hole_filling": "cuda_extension_ok",
       "sam2_file": sam2.__file__}
out = Path(DRIVE_DATA_ROOT) / "outputs" / "sam2_run_environment.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(env, indent=1))
print("recorded", out)

In [ ]:
!bash scripts/get_checkpoints.sh

In [ ]:
!python -m src.eval.get_evaluators

## Optional: read frames from the local disk
Reading thousands of JPEGs from Drive is slow. This cell copies the DAVIS and SA-V folders to the Colab disk and points `SAM2FR_DATA_ROOT` at a local folder whose `outputs` and `checkpoints` are links to Drive, so results and weights still live on Drive. Needs enough free disk (check with `!df -h /content`). Set `USE_LOCAL_COPY = False` to skip it. Safe to re-run (rsync only copies what is missing).

In [ ]:
USE_LOCAL_COPY = True  # @param {type:"boolean"}
import os, subprocess, sys
from pathlib import Path
sys.path.insert(0, ".")
from src.sam2_runner.predictor import load_models_config
from src.sam2_runner.smoke import load_data_config

os.environ["SAM2FR_DATA_ROOT"] = DRIVE_DATA_ROOT
drive_cfg = load_data_config()
if USE_LOCAL_COPY:
    local = Path("/content/sam2fr_data")
    for rel in (drive_cfg.davis.root, drive_cfg.sav.root / drive_cfg.sav.val_split, drive_cfg.sav.root / drive_cfg.sav.test_split):
        (local / rel).mkdir(parents=True, exist_ok=True)
        subprocess.run(["rsync", "-a", f"{Path(DRIVE_DATA_ROOT) / rel}/", f"{local / rel}/"], check=True)
    for rel in (drive_cfg.output_root, Path(load_models_config().checkpoint_dir)):
        (Path(DRIVE_DATA_ROOT) / rel).mkdir(parents=True, exist_ok=True)
        link = local / rel
        if not link.is_symlink():
            link.symlink_to(Path(DRIVE_DATA_ROOT) / rel)
    os.environ["SAM2FR_DATA_ROOT"] = str(local)
cfg = load_data_config()
PROMPTS_DIR = cfg.data_root / cfg.output_root / "prompts"
print("data root:", cfg.data_root)

## Prompts
One box per object at its own first visible annotated frame. Objects that are never visible are listed and left out. `sav_test` is exactly the 60-video subset in `configs/splits/sav_test_subset60.txt`.

In [ ]:
!python -m experiments.sam2_runs.make_prompts --split davis_val
!python -m experiments.sam2_runs.make_prompts --split sav_val
!python -m experiments.sam2_runs.make_prompts --split sav_test

## Pilot: an object that appears late
Before the six runs, Tiny runs on one SA-V val video that has an object whose prompt frame is after frame 0 (picked automatically from the prompts file). It writes to the separate method `sam2_tiny_pilot`, not `sam2_tiny`. The next cell prints the object's area and object score for the frames just before and after its start.

**Check:** before its start frame the late object must be empty (area 0) with a NaN score, and from its start frame it must be non-empty. If not, stop here and send back the whole output of the two pilot cells.

In [ ]:
import json, subprocess, sys
import pandas as pd

sav_prompts = json.loads((PROMPTS_DIR / "sav_val.json").read_text())
PILOT_VIDEO, PILOT_OBJ, PILOT_START = next(
    (name, int(obj), box["frame_index"])
    for name, entry in sav_prompts["videos"].items()
    for obj, box in entry["boxes"].items()
    if box["frame_index"] > 0
)
print("pilot video", PILOT_VIDEO, "| late object", PILOT_OBJ, "| starts at frame", PILOT_START)
pilot_file = PROMPTS_DIR / "sav_val_pilot.json"
pilot_file.write_text(json.dumps({"split": "sav_val", "videos": {PILOT_VIDEO: sav_prompts["videos"][PILOT_VIDEO]}}))
!python -m src.sam2_runner.run --model tiny --method sam2_tiny_pilot --prompts {pilot_file} {OFFLOAD}

In [ ]:
signals = pd.read_csv(cfg.data_root / cfg.output_root / "sam2_tiny_pilot" / "sav_val" / "signals" / f"{PILOT_VIDEO}.csv")
late = signals[(signals.object_id == PILOT_OBJ) & signals.frame_index.between(PILOT_START - 3, PILOT_START + 3)]
print(late[["frame_index", "area", "object_score", "empty"]].to_string(index=False))
before = signals[(signals.object_id == PILOT_OBJ) & (signals.frame_index < PILOT_START)]
after = signals[(signals.object_id == PILOT_OBJ) & (signals.frame_index >= PILOT_START)]
print("before start: all empty =", bool(before["empty"].all()), "| all NaN score =", bool(before.object_score.isna().all()))
print("from start: mean area =", float(after.area.mean()))

## Runs
Six cells, one per model x split. `OFFLOAD` keeps the decoded frames on the CPU; set it to `--offload-video` if a long SA-V video runs out of GPU memory (the masks are the same). Outputs go to `outputs/sam2_<model>/<split>/` on Drive.

In [ ]:
OFFLOAD = ""  # @param ["", "--offload-video"]

In [ ]:
!python -m src.sam2_runner.run --model tiny --prompts {PROMPTS_DIR}/davis_val.json {OFFLOAD}

In [ ]:
!python -m src.sam2_runner.run --model tiny --prompts {PROMPTS_DIR}/sav_val.json {OFFLOAD}

In [ ]:
!python -m src.sam2_runner.run --model tiny --prompts {PROMPTS_DIR}/sav_test.json {OFFLOAD}

In [ ]:
!python -m src.sam2_runner.run --model large --prompts {PROMPTS_DIR}/davis_val.json {OFFLOAD}

In [ ]:
!python -m src.sam2_runner.run --model large --prompts {PROMPTS_DIR}/sav_val.json {OFFLOAD}

In [ ]:
!python -m src.sam2_runner.run --model large --prompts {PROMPTS_DIR}/sav_test.json {OFFLOAD}

## Check, score, collect
The completeness check must report no missing videos for all six runs. Scoring refuses a run that is not complete.

In [ ]:
import subprocess, sys
check = subprocess.run([sys.executable, "-m", "src.eval.check_complete"], capture_output=True, text=True)
print(check.stdout, check.stderr)
assert check.returncode == 0, "Some runs are incomplete: re-run the run cell(s) named above, then this cell."


In [ ]:
import subprocess, sys
score = subprocess.run([sys.executable, "-m", "experiments.sam2_runs.score_all"], capture_output=True, text=True)
print(score.stdout, score.stderr)
assert score.returncode == 0, "Scoring was refused for some run: see the message above."


Copy the CSVs listed below (`sam2_*.csv` and `failure_stats.csv`) into the repository's `results/` folder (download them from the Colab file browser, or from the Drive copy), commit them with the usual project commit message, and tell the other person.

In [ ]:
import subprocess, sys
stats = subprocess.run([sys.executable, "-m", "experiments.sam2_runs.failure_stats", "--out", "results/failure_stats.csv"], capture_output=True, text=True)
print(stats.stdout, stats.stderr)
assert stats.returncode == 0, "Failure counts were refused for some run: see the message above."


Copy the CSVs listed below into the repository's `results/` folder (download them from the Colab file browser, or from the Drive copy), commit them with the usual project commit message, and tell the other person.

In [ ]:
import shutil
from pathlib import Path
files = sorted(Path("results").glob("sam2_*.csv")) + sorted(Path("results").glob("failure_stats.csv"))
backup = Path(DRIVE_DATA_ROOT) / "outputs" / "results_sam2"
backup.mkdir(parents=True, exist_ok=True)
for f in files:
    shutil.copy(f, backup / f.name)
    print(f, f.stat().st_size, "bytes")
print(f"{len(files)} files (expected 13: 6 score files, 6 timing files, failure_stats.csv); copies on Drive: {backup}")